# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [39]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0, 0)
        self.walls = {(0,3),(1,1),(2,4),(4,2)}
        self.slippery_states = {(1,2),(2,1),(3,3)}

        self.terminal_states = {
            (0,5): 10,
            (2,2): 2,
            (3,5): -10,
        }

        self.danger_states = {
            (1,4), (4,1)
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r,c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [
            (r,c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r,c))
        ]
        

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        elif state in self.danger_states:
            return -3
        return self.living_reward

    def get_transition_probs(self, state, action):
        if state in self.slippery_states:
            probs = (0.60,0.20,0.20)
        else:
            probs = (0.90, 0.05, 0.05)
        
        dr, dc = action
        moves = [(dr, dc), (dc, -dr), (-dc,dr)]
        
        next_states_with_probability = {}
        for move, prob in zip(moves, probs):
            nr, nc = state[0] + move[0], state[1] + move[1]
            next_state = (nr, nc)
            if not self.is_valid_state(next_state):
                next_state = state
            next_states_with_probability[next_state] = next_states_with_probability.get(next_state, 0) + prob
        return list(next_states_with_probability.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [40]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [41]:
def expected_next_value(grid, state, action, V):
    transitions = grid.get_transition_probs(state, action)
    expected_val = 0
    for next_state, prob in transitions:
        expected_val += prob * V[next_state]
    return expected_val


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {}
    for state in grid.states():
        V[state] = 0

    for i in range(max_iter):
        V_new = {}
        
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
                continue
            else:
                reward = grid.get_reward(state)
                val_in_future = []
                for action in grid.actions:
                    expected_val = expected_next_value(grid, state, action, V)
                    val_in_future.append(expected_val)
                
                best_future = max(val_in_future)
                V_new[state] = reward + grid.gamma * best_future
        delta = max(abs(V_new[state] - V[state]) for state in grid.states())
        V = V_new
        if delta < threshold:
            print(f"Value Iteration convergió en {i + 1} iteraciones")
            return V, i + 1
    print(f"Value Iteration alcanzó máximo de iteraciones")
    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    
    for state in grid.states():
        if grid.is_terminal(state):
            policy[state] = None
            continue
        action_values = []
        for action in grid.actions:
            #expected value of applying this action in the current state
            expected_val = expected_next_value(grid, state, action, V)
            action_values.append((action, expected_val))

        #Select the option with the highest expected value
        best_action = max(action_values, key=lambda x: x[1])[0]
        policy[state] = best_action

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [42]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {}
    for state in grid.states():
        V[state] = 0
    
    for i in range(max_iter):
        V_new = {}
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                reward = grid.get_reward(state)
                expected_val = expected_next_value(grid, state, action, V)
                V_new[state] = reward + grid.gamma * expected_val

        delta = max(abs(V_new[state] - V[state]) for state in grid.states())
        V = V_new
        if delta < threshold:
            return V
    return V


def policy_improvement(grid, V):
    policy_new = {}
    
    for state in grid.states():
        if grid.is_terminal(state):
            policy_new[state] = None
            continue
        
        best_value = float('-inf')
        best_action = None
        
        for action in grid.actions:
            expected_val = expected_next_value(grid, state, action, V)
            if expected_val > best_value:
                best_value = expected_val
                best_action = action
        
        policy_new[state] = best_action
        
    return policy_new


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {}
    for state in grid.states():
        if not grid.is_terminal(state):
            policy[state] = grid.actions[0]
        else:
            policy[state] = None
    
    history = []
    
    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold=threshold)
        
        policy_new = policy_improvement(grid, V)
        
        policy_changed = any(policy[state] != policy_new[state] for state in grid.states() if not grid.is_terminal(state))
        history.append({'iteration': _, 'policy_changed': policy_changed})
        
        policy = policy_new
        if not policy_changed:
            return policy, V, history
    return policy, V, history


## Parte 4 — Visualización y comparación


In [43]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [44]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


Value Iteration convergió en 20 iteraciones
=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'iteration': 0, 'policy_changed': True}, {'iteration': 1, 'policy_changed': True}, {'iteration': 2, 'policy_changed': True}, {'iteration': 3, 'policy_changed': False}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
Porque si preferimos el +10 aunque esté más lejos, tendremos mayor ganancia y esto compensará el costo extra de los pasos. A diferencia de si agarramos la carga de +2, llegaremos más rápido pero tendremos una ganancia mucho menor.

+10 LEJANO: -8 (pasos) + 10 (recompensa) = +2 
+2 CERCANO: -5 (pasos) + 2 (recompensa) = -3 
Gana el +10

2. ¿Por qué una recompensa menor podría ser óptima?
Si ponemos cada paso más caro, el hecho de ir más lejos va a costar muchísimo más que ir al nodo terminal más cercano aunque nos dé una recompensa más pequeña. Aquí se valoraría más el hecho de que esté cerca.

3. ¿En qué estados el piso resbaloso cambia la decisión?
En ningún estado el piso resbaloso cambia la dirección óptima. En todos los casos, pese a que el robot tiene solo el 60% del control, la mejor dirección sigue siendo la misma. Pero, esto depende de la política específica. Si nosotros cambiamos el living_reward o la probabilidad de movimiento, podríamos encontrar estados donde el piso resbaloso cambia la dirección óptima.

4. ¿Qué papel cumple el costo por paso `-1`?
Si no existe el costo por paso el algoritmo se volvería perezoso y no habría nada que lo haga moverse, así que podría quedarse en el nodo inicial infinitamente por no querer moverse. El hecho de que le duela vivir es lo que hace que deba tomar decisiones y moverse.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

Porque el mundo no es homogéneo por donde lo mires, es decir, hay lugares con piso resbaloso y hay lugares con piso normal. Si usaramos la misma probabilidad en todos lados, ignoraríamos esa diferencia haciendo que el juego carezca de sentido

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
